[![DOI](https://zenodo.org/badge/854202210.svg)](https://doi.org/10.5281/zenodo.18888945)
[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/ruzayqat/LSMCMC/HEAD?labpath=LSMCMC.ipynb)

# Two Localization Strategies for Sequential MCMC Data Assimilation with Applications to Nonlinear Non-Gaussian Geophysical Models

**Authors:** Hamza Ruzayqat, Hristo G. Chipilski, and Omar Knio

**Code author & maintainer:** Hamza Ruzayqat — [ruzayqath@gmail.com](mailto:ruzayqath@gmail.com) | GitHub: [@ruzayqat](https://github.com/ruzayqat)

**Date created:** March 2026  
**Repository:** [https://github.com/ruzayqat/LSMCMC](https://github.com/ruzayqat/LSMCMC)  
**License:** MIT (see `LICENSE`)

---

This notebook serves two purposes:

1. **Running guide** — How to run the LSMCMC (Variants 1 and 2), choose MCMC kernels for nonlinear/non-Gaussian observation models, and run LETKF, including parallel execution.
2. **Figure reproduction** — Regenerate all 19 paper figures from pre-computed output.

## Table of Contents

### Part I — Running Guide
- [Quick Start](#Part-I-—-Running-Guide)
- [Running LSMCMC — Variant 1](#Running-LSMCMC-—-Variant-1-(Block-Partition))
- [Running LSMCMC — Variant 2](#Running-LSMCMC-—-Variant-2-(Halo-with-Gaspari–Cohn-Tapering))
- [Choosing the MCMC Kernel](#Choosing-the-MCMC-Kernel-(Nonlinear-/-Non-Gaussian-Case))
- [Running LETKF](#Running-LETKF)
- [Linear Gaussian Experiment](#Linear-Gaussian-Experiment-(Standalone))
- [MLSWE Data Preparation](#MLSWE-Experiments-—-Data-Preparation)

### Part II — Figure Reproduction
- [1. Import Libraries](#1.-Import-Required-Libraries)
- [2. Global Plot Styling](#2.-Global-Plot-Styling-and-Configuration)
- [3. Helper Functions](#3.-Helper-Functions)
- [4. Shared Plotting Functions](#4.-Shared-Plotting-Functions)
- [Figure 1 — Localization Illustration](#Figure-1-—-Localization-Illustration-(V1-vs-V2))
- [Figures 2–6 — Linear Gaussian](#Figures-2–6-—-Linear-Gaussian-Experiment-($d-=-14{,}400$))
- [Figures 7–9 — MLSWE Linear Obs](#Figures-7–9-—-MLSWE-Linear-Observation-Model)
- [Figures 10–12 — Timeseries (V2)](#Figures-10–12-—-Velocity-/-SST-/-SSH-Timeseries-(V2-Linear))
- [Figures 13–14 — NL Twin](#Figures-13–14-—-MLSWE-Nonlinear-Twin-Experiment)
- [Figures 15–19 — Cauchy Noise Twin](#Figures-15–19-—-arctan-+-Cauchy-Noise-Twin-Experiment)
- [Summary & Verification](#Summary)
- [Environment Info](#Environment-Information)
- [How to Cite](#How-to-Cite)

## Repository Structure

| Path | Description |
|------|-------------|
| `mlswe/` | Python package: 3-layer MLSWE model, LSMCMC V1/V2, NL-LSMCMC V1/V2, LETKF |
| `linear_gaussian/` | Self-contained linear Gaussian experiment (KF, LSMCMC, LETKF) |
| `run_mlswe_lsmcmc_*.py` | Runner scripts for LSMCMC data assimilation |
| `run_mlswe_letkf_*.py` | Runner scripts for LETKF data assimilation |
| `example_input_*.yml` | YAML configuration files for all experiments |
| `data/` | HYCOM reanalysis, OSMC drifter obs, SWOT SSH, bathymetry |
| `output_*/` | Pre-computed DA output (NetCDF) |
| `paper_figures/` | Generated figures (PDF/PNG) |

## Dependencies

```
numpy, scipy, matplotlib, netCDF4, pyyaml
```

Install with: `pip install -r requirements.txt`  
Or with Conda: `conda env create -f environment.yml && conda activate lsmcmc`

# Part I — Running Guide

## Quick Start

Every experiment follows the same pattern:
```bash
python3 run_mlswe_lsmcmc_<variant>.py  [config.yml]
```
If no config is given, a sensible default is used. All configuration is in YAML.

## The Four Experiment Categories

| # | Experiment | State Model | Observation Model | Scripts | Config |
|---|------------|-------------|-------------------|---------|--------|
| 1 | **Linear Gaussian** | $z_{t_{k+1}} = A z_{t_k} + \varepsilon_{t_k}$ | $y_{t_k} = Hz_{t_k} + \nu_{t_k}$ | `linear_gaussian/linear_forward_run_*.py` | `linear_gaussian/input_linear_*.yml` |
| 2 | **MLSWE linear obs** | $z_{t_{k+1}} = \Phi(z_{t_k}) + \varepsilon_{t_k}$ | $y_{t_k} = Hz_{t_k} + \nu_{t_k}$ | `run_mlswe_lsmcmc_ldata_V{1,2}.py` | `example_input_mlswe_ldata_V{1,2}.yml` |
| 3 | **MLSWE NL twin** | $z_{t_{k+1}} = \Phi(z_{t_k}) + \varepsilon_{t_k}$ | $y_{t_k} = \arctan(Hz_{t_k}) + \nu_{t_k}$ | `run_mlswe_lsmcmc_nldata_V{1,2}_twin.py` | `example_input_mlswe_nldata_V{1,2}_twin.yml` |
| 4 | **Cauchy noise twin** | $z_{t_{k+1}} = \Phi(z_{t_k}) + \varepsilon_{t_k}$ | $y_{t_k} = \arctan(Hz_{t_k}) + t_\nu$ | `nlgamma_ldata/run_nlgamma_twin*.py` | `nlgamma_ldata/input_nlgamma_*.yml` |

where $\varepsilon_{t_k} \sim \mathcal{N}(0, Q)$ and $\nu_{t_k} \sim \mathcal{N}(0, R)$ (Gaussian), $t_\nu$ is Cauchy-distributed, and $\Phi$ is the MLSWE forward model.


## Running LSMCMC — Variant 1 (Block Partition)

**V1** partitions the full mesh into $\Gamma$ non-overlapping rectangular blocks. The posterior is then limited to the combined observed blocks (i.e., blocks that contain at least one observation).

When the observation operator is **linear** and noise is **Gaussian**, LSMCMC uses **exact sampling from the local Gaussian posterior** — no MCMC chain is needed. This is used by `run_mlswe_lsmcmc_ldata_V{1,2}.py` and the linear-Gaussian example `linear_forward_run_lsmcmc_{v1,v2}.py`.

```bash
# Linear Gaussian experiment (standalone, no external data)
cd linear_gaussian/
python3 linear_forward_run_lsmcmc_v1.py

# Linear observation model (real drifter + SWOT data)
python3 run_mlswe_lsmcmc_ldata_V1.py  example_input_mlswe_ldata_V1.yml

# Nonlinear (arctan) observation model — twin experiment
python3 run_mlswe_lsmcmc_nldata_V1_twin.py  example_input_mlswe_nldata_V1_twin.yml
```

**Key config parameters for V1:**
```yaml
nforecast: 25          # forecast ensemble size N_f
mcmc_N: 500            # posterior samples per cycle N_a
burn_in: 500           # MCMC burn-in
num_subdomains: 50     # Γ — number of partition blocks
```

## Running LSMCMC — Variant 2 (Halo with Gaspari–Cohn Tapering)

**V2** uses overlapping halos with Gaspari–Cohn distance-based tapering, analogous to LETKF's localization. Each observed block is augmented with a halo of neighboring cells.

```bash
# Linear Gaussian experiment (standalone, no external data)
cd linear_gaussian/
python3 linear_forward_run_lsmcmc_v2.py

# Linear observation model
python3 run_mlswe_lsmcmc_ldata_V2.py  example_input_mlswe_ldata_V2.yml

# Nonlinear (arctan) twin experiment
python3 run_mlswe_lsmcmc_nldata_V2_twin.py  example_input_mlswe_nldata_V2_twin.yml
```

V2 scripts are thin wrappers around V1: they swap the filter class (`Loc_SMCMC_MLSWE_Filter_V2`) and load a V2 config. The YAML structure is identical to V1, but V2 additionally uses a **localization radius** (halo extent) set internally by the filter.


## Choosing the MCMC Kernel (Nonlinear / Non-Gaussian Case)

When the observation operator is **linear** and noise is **Gaussian**, LSMCMC uses **exact sampling from the local Gaussian posterior** — no MCMC chain is needed. This is used by `run_mlswe_lsmcmc_ldata_V{1,2}.py`.

When the observation model is **nonlinear** (e.g., $y = \arctan(Hz) + \varepsilon$) or the noise is **non-Gaussian** (e.g., Student-$t$ / Cauchy), the local posterior is intractable and an MCMC kernel is required. The YAML config controls this:

```yaml
# MCMC kernel selection (in the NL config files)
mcmc_kernel: 'pcn'          # Options: 'gibbs_mh', 'pcn', 'mala', 'hmc'
mcmc_step_size: 0.5         # step size (pcn: β, MALA/HMC: ε)
mcmc_adapt: true            # adaptive step-size tuning
mcmc_target_acc: 0.40       # target acceptance rate for adaptation
pcn_beta: 0.3               # pCN-specific: proposal correlation parameter
hmc_leapfrog_steps: 10      # HMC-specific: leapfrog integration steps
```

### Available Kernels

| Kernel | `mcmc_kernel` | Best For | Notes |
|--------|--------------|----------|-------|
| **Gibbs-MH** | `'gibbs_mh'` | Moderate nonlinearity | Component-wise Metropolis–Hastings |
| **pCN** | `'pcn'` | General nonlinear | Preconditioned Crank–Nicolson; dimension-robust |
| **MALA** | `'mala'` | Smooth likelihoods | Metropolis-adjusted Langevin; uses gradient |
| **HMC** | `'hmc'` | Strong nonlinearity | Hamiltonian Monte Carlo; best mixing but costliest |

**Recommendation from the paper:**
- For moderate arctan nonlinearity with Gaussian noise → **pCN** (`mcmc_kernel: 'pcn'`)
- For arctan + Cauchy (heavy-tailed) noise → **pCN** works well; **HMC** gives the best RMSE at higher cost
- The V1 (HMC) variant in the NL twin experiment uses `mcmc_kernel: 'hmc'` with `hmc_leapfrog_steps: 10`

## Running LETKF

The **Local Ensemble Transform Kalman Filter** (LETKF) serves as the baseline. It uses Gaspari–Cohn localization and RTPS (Relaxation-to-Prior-Spread) inflation.

```bash
# Linear observations (real drifter + SWOT)
python3 run_mlswe_ldata_letkf_mpi.py  example_input_mlswe_letkf_best.yml

# Nonlinear (arctan) twin experiment
python3 run_mlswe_letkf_nl_twin.py  example_input_mlswe_letkf_nl_twin.yml
```

**Key LETKF config parameters:**
```yaml
nanals: 25                 # ensemble size K
hcovlocal_scale: 200       # Gaspari–Cohn localization radius (km)
covinflate1: 0.95          # RTPS relaxation coefficient (0=no inflation, 1=full)
```

## Parallel Execution

All scripts parallelize using Python's `multiprocessing` (not MPI). The number of parallel workers is controlled by:

```yaml
ncores: 52       # total CPU cores available
```

**LSMCMC V1** parallelizes in three ways:
1. **Forecast parallelism** — the $N_f$ forecast ensemble members run on separate processes.
2. **$M$-run averaging** — run $M$ independent LSMCMC filters and average (variance reduction).
3. **MCMC chain splitting** — a long MCMC chain can be split into multiple shorter parallel chains (useful when the number of iterations is much larger than burn-in, since all parallel chains share the same burn-in period).

**LSMCMC V2** parallelizes in three ways:
1. **Forecast parallelism** — as in V1.
2. **$M$-run averaging** — as in V1.
3. **Block parallelism** — loop over all observed blocks in parallel.

**LETKF** parallelizes in two ways:
1. **Forecast parallelism** — the $K$ ensemble forecasts run across `ncores` processes.
2. **Grid-point parallelism** — loop over all grid points in parallel during the analysis step.

> **Tip:** Set `ncores` to match your machine's physical core count. Pin BLAS threads to 1 per process to avoid thread oversubscription:
> ```bash
> export OMP_NUM_THREADS=1
> export MKL_NUM_THREADS=1
> python3 run_mlswe_lsmcmc_ldata_V1.py  config.yml
> ```


## Linear Gaussian Experiment (Standalone)

The `linear_gaussian/` directory is **fully self-contained** — no external data download is needed.

### Data Generation

The synthetic truth and observations are generated procedurally:

```bash
cd linear_gaussian/

# 1. Generate synthetic truth trajectory and SWOT-like swath observations
python3 linear_forward_generate_data.py
# → Saves: linear_gaussian/linear_gaussian_data.npz
#   Contains: truth Z_true (100×14400), observation indices C_k,
#             obs values Y_k, and noise parameters.
```

This script creates a $120 \times 120$ grid ($d = 14{,}400$) with known linear dynamics ($Z_{k+1} = 0.25\,Z_k + 0.05\,W_k$) and dual SWOT-like swath observations that sweep across the domain over 20-cycle periods. The exact Kalman filter posterior is available for verification.

**All other scripts in `linear_gaussian/` read from `linear_gaussian_data.npz`:**

```bash
# 2. Run exact Kalman filter (reference solution)
python3 linear_forward_run_kf.py

# 3. Run LSMCMC V1 (block partition, Γ=900)
python3 linear_forward_run_lsmcmc_v1.py

# 4. Run LSMCMC V2 (halo + Gaspari–Cohn)
python3 linear_forward_run_lsmcmc_v2.py

# 5. Run LETKF (K=50, multiprocessing)
python3 linear_forward_run_letkf_mpi.py --config input_linear_letkf.yml

# 6. LETKF sensitivity sweep (localization × inflation)
python3 linear_forward_run_letkf_sensitivity.py
```

## MLSWE Experiments — Data Preparation

The MLSWE experiments (linear obs, nonlinear twin, nonlinear real-data, Cauchy twin) **all share the same underlying data** stored in `data/`. Since this data is ~1.8 GB (too large for GitHub), it must be downloaded separately.

### Step 0: Download All Data (One-Time Setup)

A single script downloads everything needed:

```bash
python3 download_all_data.py
```

This downloads (total ~200–600 MB):
1. **HYCOM boundary conditions** — SSH, velocity, SST via OPeNDAP → `data/hycom_bc_2024aug.nc`
2. **ETOPO bathymetry** — ocean depth via OPeNDAP → `data/etopo_bathy_*.npy`
3. **OSMC drifter observations** — position + SST via ERDDAP → `data/osmc_drifters_2024-08-01_2024-08-11.csv`
4. **SWOT L2 SSH** — satellite altimetry via NASA Earthdata → `data/swot_2024aug_new/SWOT_*.nc`
5. **HYCOM SSH reference** — extracted from BC file → `data/hycom_ssh_ref_*.npy`
6. **HYCOM SST reference** — extracted from BC file → `data/hycom_sst_ref_*.npy`

> **Note:** SWOT download requires `pip install earthaccess` and a free [NASA Earthdata](https://urs.earthdata.nasa.gov/) account. Use `--skip-swot` to skip it if you only need drifter data.

### Step 1: Process Observations

After downloading raw data, process it into the format expected by the runner scripts:

```bash
python3 generate_drifter_obs.py       # → data/obs_2024aug/swe_drifter_obs.nc
python3 prebin_swot_ssh.py            # → data/swot_2024aug_new/swot_ssh_binned_80x70.nc
python3 generate_synthetic_swot.py    # → data/obs_2024aug/swot_ssh_obs_with_synthetic.nc
```

Once `data/` is populated, **all four MLSWE experiment types** (linear, NL twin, NL real-data, Cauchy) can access it without further downloads.

## Running a Full MLSWE Experiment End-to-End

```bash
# Step 2: Run LSMCMC V1 (linear observations)
python3 run_mlswe_lsmcmc_ldata_V1.py  example_input_mlswe_ldata_V1.yml
# → output in output_lsmcmc_ldata_V1/mlswe_lsmcmc_out.nc

# Step 3: Run LSMCMC V2 (linear observations)
python3 run_mlswe_lsmcmc_ldata_V2.py  example_input_mlswe_ldata_V2.yml
# → output in output_lsmcmc_ldata_V2/mlswe_lsmcmc_out.nc

# Step 4: Run LETKF baseline
python3 run_mlswe_ldata_letkf_mpi.py  example_input_mlswe_letkf_best.yml
# → output in output_letkf_ldata_K25/mlswe_letkf_out.nc

# Step 5: Generate all paper figures (single command)
python3 generate_all_figures.py
# Runs generate_paper_figures, generate_nlgamma_figures, and
# regen_timeseries_1x2 in sequence → saves to paper_figures/
```

---

# Part II — Figure Reproduction

The sections below regenerate all 19 paper figures using the pre-computed output from the experiments above. Figures are saved to `paper_figures/` and displayed inline.

## 0. Download Pre-Computed Output Data (Binder / Quick Start)

If you're running on **Binder** or want to reproduce figures without running the full DA experiments (which take hours), download the pre-computed output data first.

> **Note:** The **Linear Gaussian experiment** (Figures 2–6) is self-contained and runs quickly (~5 min). MLSWE figures require either downloaded data OR running the full DA experiments locally (see paper for timing details).

In [ ]:
# Download pre-computed output data (skip if already present or running locally with your own data)
import subprocess
import sys
from pathlib import Path

BASEDIR = Path('.').resolve()
output_marker = BASEDIR / 'output_lsmcmc_ldata_V1' / 'mlswe_lsmcmc_out.nc'

if not output_marker.exists():
    print("Pre-computed output data not found. Attempting download...")
    result = subprocess.run([sys.executable, 'download_output_data.py', '--noninteractive'], 
                          capture_output=True, text=True)
    if result.returncode == 0:
        print("Download complete!")
    else:
        print("Download skipped or failed. Some figures may not work.")
        print("You can run experiments locally or check download_output_data.py for details.")
        if result.stderr:
            print(f"Details: {result.stderr[:500]}")
else:
    print(f"Output data found: {output_marker.parent}")

## 1. Import Required Libraries

In [ ]:
import os
import sys
import json
import glob
import re

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from matplotlib.colors import Normalize
from scipy.ndimage import gaussian_filter, distance_transform_edt
from scipy.interpolate import RegularGridInterpolator
from netCDF4 import Dataset
from IPython.display import Image, display

# ── Paths (define early so all cells can use them) ─────────────────
BASEDIR = os.path.dirname(os.path.abspath('LSMCMC.ipynb'))
OUTDIR  = os.path.join(BASEDIR, 'paper_figures')
os.makedirs(OUTDIR, exist_ok=True)

# Print versions for reproducibility
print(f"NumPy:      {np.__version__}")
print(f"Matplotlib: {matplotlib.__version__}")
print(f"Python:     {sys.version}")
print(f"BASEDIR:    {BASEDIR}")

%matplotlib inline

## 2. Global Plot Styling and Configuration

Set publication-quality defaults matching the paper's visual style.

In [ ]:
# ── Publication-quality global settings ─────────────────────────────
plt.rcParams.update({
    'font.size': 16,
    'axes.titlesize': 18,
    'axes.labelsize': 16,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
    'legend.fontsize': 13,
    'figure.titlesize': 20,
    'lines.linewidth': 2.0,
    'axes.linewidth': 1.2,
    'xtick.major.width': 1.0,
    'ytick.major.width': 1.0,
    'xtick.major.size': 5,
    'ytick.major.size': 5,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.1,
})

# Standard color palette
COLORS = {
    'V1': '#1f77b4',
    'V2': '#ff7f0e',
    'LETKF': '#2ca02c',
    'V1 HMC': '#d62728',
    'KF': 'gray',
}

def savefig(fig, name):
    """Save figure to paper_figures/ in PDF (and PNG for raster)."""
    path_pdf = os.path.join(OUTDIR, name)
    fig.savefig(path_pdf, bbox_inches='tight', dpi=300)
    if name.endswith('.pdf'):
        path_png = path_pdf.replace('.pdf', '.png')
        fig.savefig(path_png, bbox_inches='tight', dpi=150)
    print(f"  ✓ Saved: {path_pdf}")

print(f"Figures will be saved to: {OUTDIR}")

## 3a. Helper Functions — Data Loading and Field Extraction

Utilities for loading MLSWE/LETKF NetCDF output and extracting physical fields (SSH, SST).

In [ ]:
def _fill_nan_nearest(arr_2d):
    """Fill NaN in a 2-D array with nearest finite value."""
    mask = np.isnan(arr_2d)
    if not mask.any():
        return arr_2d
    if mask.all():
        arr_2d[:] = 0.0
        return arr_2d
    ind = distance_transform_edt(mask, return_distances=False, return_indices=True)
    arr_2d[mask] = arr_2d[tuple(ind[:, mask])]
    return arr_2d


def load_mlswe_analysis(nc_path):
    """Load MLSWE analysis output. Returns data dict."""
    with Dataset(nc_path, 'r') as nc:
        raw = np.asarray(nc.variables['lsmcmc_mean'][:])
        rmse_vel = np.asarray(nc.variables['rmse_vel'][:])
        rmse_sst = np.asarray(nc.variables['rmse_sst'][:])
        rmse_ssh = np.asarray(nc.variables['rmse_ssh'][:]) if 'rmse_ssh' in nc.variables else None
        obs_times = np.asarray(nc.variables['obs_times'][:]) if 'obs_times' in nc.variables else None
        ny = int(nc.ny) if hasattr(nc, 'ny') else raw.shape[3]
        nx = int(nc.nx) if hasattr(nc, 'nx') else raw.shape[4]
        H_b = np.asarray(nc.variables['H_b'][:]) if 'H_b' in nc.variables else None
    return {
        'raw': raw, 'rmse_vel': rmse_vel, 'rmse_sst': rmse_sst,
        'rmse_ssh': rmse_ssh, 'obs_times': obs_times, 'ny': ny, 'nx': nx, 'H_b': H_b,
    }


def load_mlswe_letkf(nc_path):
    """Load LETKF output from NetCDF."""
    with Dataset(nc_path, 'r') as nc:
        rmse_vel = np.asarray(nc.variables['rmse_vel'][:])
        rmse_sst = np.asarray(nc.variables['rmse_sst'][:])
        rmse_ssh = np.asarray(nc.variables['rmse_ssh'][:]) if 'rmse_ssh' in nc.variables else None
    return {'rmse_vel': rmse_vel, 'rmse_sst': rmse_sst, 'rmse_ssh': rmse_ssh}


def extract_ssh(data, cycle, H_b=None):
    """Extract SSH anomaly at a given cycle."""
    raw = data['raw']
    ny, nx = data['ny'], data['nx']
    h_total = raw[cycle, :, 0, :, :].sum(axis=0)
    if H_b is not None:
        ssh = h_total - H_b.reshape(ny, nx)
    else:
        ssh = h_total - 4000.0
    return ssh


def extract_sst(data, cycle):
    """Extract SST at a given cycle (Kelvin)."""
    return data['raw'][cycle, 0, 3, :, :]


def smooth_ocean(field, H_b, sigma=2.0):
    """Gaussian-smooth an ocean field, masking land (H_b < 200 m)."""
    ocean_mask = H_b >= 200.0 if H_b is not None else np.ones_like(field, dtype=bool)
    filled = field.copy()
    if ocean_mask.any():
        filled[~ocean_mask] = np.nanmean(field[ocean_mask])
    else:
        filled[~ocean_mask] = 0.0
    smoothed = gaussian_filter(filled, sigma=sigma)
    return np.ma.masked_where(~ocean_mask, smoothed)

print("Data I/O helpers defined.")

## 3b. Helper Functions — HYCOM Reanalysis Loading

Utilities for loading and interpolating HYCOM boundary-condition reanalysis data onto the model grid.

In [ ]:
def load_hycom_reanalysis(bc_path, model_lon, model_lat, model_time_sec):
    """Load HYCOM reanalysis and interpolate to model grid at a given time."""
    with Dataset(bc_path, 'r') as nc:
        bc_lat = np.asarray(nc.variables['lat'][:], dtype=np.float64)
        bc_lon = np.asarray(nc.variables['lon'][:], dtype=np.float64)
        bc_times = np.asarray(nc.variables['time'][:], dtype=np.float64)
        bc_ssh = np.asarray(nc.variables['ssh'][:], dtype=np.float64)
        bc_uo  = np.asarray(nc.variables['uo'][:],  dtype=np.float64)
        bc_vo  = np.asarray(nc.variables['vo'][:],  dtype=np.float64)
        bc_sst = np.asarray(nc.variables['sst'][:], dtype=np.float64)
    for arr3d in [bc_ssh, bc_uo, bc_vo, bc_sst]:
        for t in range(arr3d.shape[0]):
            _fill_nan_nearest(arr3d[t])
    if np.nanmean(bc_sst) < 100.0:
        bc_sst += 273.15
    t_idx = np.interp(model_time_sec, bc_times, np.arange(len(bc_times)))
    t_lo = int(np.floor(t_idx))
    t_hi = min(t_lo + 1, len(bc_times) - 1)
    alpha = t_idx - t_lo
    result = {}
    mg_lat, mg_lon = np.meshgrid(model_lat, model_lon, indexing='ij')
    for name, field3d in [('ssh', bc_ssh), ('u', bc_uo),
                          ('v', bc_vo), ('sst', bc_sst)]:
        snap = ((1 - alpha) * field3d[t_lo] + alpha * field3d[t_hi]
                if t_lo != t_hi else field3d[t_lo])
        interp_fn = RegularGridInterpolator(
            (bc_lat, bc_lon), snap,
            method='linear', bounds_error=False, fill_value=np.nan)
        result[name] = interp_fn((mg_lat, mg_lon))
    return result


def _find_bc_path():
    """Find HYCOM boundary-condition NetCDF."""
    for p in ['data/hycom_bc_2024aug.nc', 'data/hycom_bc.nc']:
        fp = os.path.join(BASEDIR, p)
        if os.path.exists(fp):
            return fp
    return None


def _load_hycom_pair(bc_path, data):
    """Load HYCOM reanalysis at first and last analysis times."""
    lon_grid = np.linspace(-60.0, -20.0, data['nx'])
    lat_grid = np.linspace(10.0, 45.0, data['ny'])
    obs_times = data['obs_times']
    hycom_init  = load_hycom_reanalysis(bc_path, lon_grid, lat_grid, obs_times[1])
    hycom_final = load_hycom_reanalysis(bc_path, lon_grid, lat_grid, obs_times[-1])
    return hycom_init, hycom_final

print("HYCOM helpers defined.")

## 4a. Plotting — RMSE Comparison (3-Panel)

Reusable RMSE time-series comparison for velocity, SST, and SSH.

In [ ]:
LON_RANGE = (-60, -20)
LAT_RANGE = (10, 45)


def plot_compare_rmse_3panel(datasets, labels, colors_list, fname,
                              title='RMSE Comparison',
                              sigma_vel=0.10, sigma_sst=0.40, sigma_ssh=0.25,
                              ssh_clip=None):
    """
    3-panel (vel / SST / SSH) RMSE comparison.
    datasets: list of dicts with 'rmse_vel', 'rmse_sst', 'rmse_ssh'.
    """
    has_ssh = any(d.get('rmse_ssh') is not None for d in datasets)
    nrows = 3 if has_ssh else 2
    fig, axes = plt.subplots(nrows, 1, figsize=(10, 3.5 * nrows), sharex=True)
    _ssh_skip = 0

    for d, lab, col in zip(datasets, labels, colors_list):
        n = len(d['rmse_vel'])
        t = np.arange(1, n + 1)
        vel = np.asarray(d['rmse_vel'], dtype=float)
        sst = np.asarray(d['rmse_sst'], dtype=float)
        ssh_arr = np.asarray(d['rmse_ssh'], dtype=float) if d.get('rmse_ssh') is not None else None
        valid = np.isfinite(vel) & np.isfinite(sst)
        if ssh_arr is not None:
            valid &= np.isfinite(ssh_arr)
        if not valid.all():
            last_valid = np.where(valid)[0][-1] + 1 if valid.any() else 0
            t = t[:last_valid]; vel = vel[:last_valid]; sst = sst[:last_valid]
            if ssh_arr is not None:
                ssh_arr = ssh_arr[:last_valid]
        axes[0].plot(t, vel, color=col, linewidth=2, label=lab)
        axes[1].plot(t, sst, color=col, linewidth=2, label=lab)
        if has_ssh and ssh_arr is not None:
            ssh_mean = float(np.nanmean(ssh_arr))
            if ssh_clip is not None and ssh_mean > ssh_clip:
                axes[2].text(0.98, 0.95 - 0.08 * _ssh_skip,
                             f'{lab}: {ssh_mean:.1f} m (off-scale)',
                             transform=axes[2].transAxes, ha='right', va='top',
                             fontsize=9, color=col,
                             bbox=dict(boxstyle='round,pad=0.3', fc='white', ec=col, alpha=0.8))
                _ssh_skip += 1
            else:
                axes[2].plot(t, ssh_arr, color=col, linewidth=2, label=lab)

    axes[0].axhline(y=sigma_vel, color='gray', ls='--', lw=1, alpha=0.5,
                    label=f'$\\sigma_{{vel}}={sigma_vel}$')
    axes[1].axhline(y=sigma_sst, color='gray', ls='--', lw=1, alpha=0.5,
                    label=f'$\\sigma_{{SST}}={sigma_sst}$')
    if has_ssh:
        axes[2].axhline(y=sigma_ssh, color='gray', ls='--', lw=1, alpha=0.5,
                        label=f'$\\sigma_{{SSH}}={sigma_ssh}$')
    axes[0].set_ylabel('Velocity RMSE (m/s)'); axes[0].set_title(title)
    axes[0].legend(loc='lower right'); axes[0].grid(True, alpha=0.3)
    axes[1].set_ylabel('SST RMSE (K)')
    axes[1].legend(loc='lower right'); axes[1].grid(True, alpha=0.3)
    if has_ssh:
        axes[2].set_ylabel('SSH RMSE (m)')
        axes[2].legend(loc='lower right'); axes[2].grid(True, alpha=0.3)
    axes[-1].set_xlabel('Assimilation Cycle')
    fig.tight_layout()
    savefig(fig, fname)

print("plot_compare_rmse_3panel defined.")

## 4b. Plotting — Field Snapshot Panels

4-row field comparison (SSH, U, V, SST) with optional HYCOM reanalysis columns.

In [ ]:
def plot_field_panels(data, label, fname,
                       hycom_init=None, hycom_final=None):
    """
    4-row field comparison: SSH, U, V, SST.
    With HYCOM: 4 rows x 3 cols (Init | Final | HYCOM Final).
    """
    from matplotlib.gridspec import GridSpec
    ny, nx = data['ny'], data['nx']
    H_b = data['H_b']
    nassim = len(data['rmse_vel'])
    lon = np.linspace(LON_RANGE[0], LON_RANGE[1], nx)
    lat = np.linspace(LAT_RANGE[0], LAT_RANGE[1], ny)
    has_hycom = hycom_init is not None and hycom_final is not None
    ncols = 3 if has_hycom else 2
    nrows = 4

    fig = plt.figure(figsize=(5 * ncols + 0.8, 4.5 * nrows))
    width_ratios = [1] * ncols + [0.04]
    gs = GridSpec(nrows, ncols + 1, figure=fig, width_ratios=width_ratios,
                  wspace=0.25, hspace=0.30)
    axes = np.empty((nrows, ncols), dtype=object)
    cbar_axes = []
    for r in range(nrows):
        for c in range(ncols):
            axes[r, c] = fig.add_subplot(gs[r, c])
        cbar_axes.append(fig.add_subplot(gs[r, ncols]))

    def _smooth(field):
        return smooth_ocean(field, H_b, sigma=2.0)

    def _plot(ax, field, title, cmap, vmin, vmax):
        ax.set_facecolor('0.88')
        im = ax.pcolormesh(lon, lat, field, cmap=cmap, vmin=vmin, vmax=vmax, shading='auto')
        ax.set_title(title, fontsize=14)
        ax.set_xlabel('Lon ($^\\circ$E)'); ax.set_ylabel('Lat ($^\\circ$N)')
        return im

    ssh_init  = extract_ssh(data, 0, H_b);  ssh_final  = extract_ssh(data, nassim, H_b)
    u_init  = data['raw'][0, 0, 1, :, :];  u_final  = data['raw'][nassim, 0, 1, :, :]
    v_init  = data['raw'][0, 0, 2, :, :];  v_final  = data['raw'][nassim, 0, 2, :, :]
    sst_init  = extract_sst(data, 0) - 273.15;  sst_final  = extract_sst(data, nassim) - 273.15

    ssh_init_s  = _smooth(ssh_init);  ssh_final_s  = _smooth(ssh_final)
    u_init_s  = _smooth(u_init);  u_final_s  = _smooth(u_final)
    v_init_s  = _smooth(v_init);  v_final_s  = _smooth(v_final)
    sst_init_s  = _smooth(sst_init);  sst_final_s  = _smooth(sst_final)

    if has_hycom:
        hycom_ssh_s = _smooth(hycom_final['ssh'])
        hycom_u_s   = _smooth(hycom_final['u'])
        hycom_v_s   = _smooth(hycom_final['v'])
        hycom_sst_s = _smooth(hycom_final['sst'] - 273.15)

    vmax_ssh = max(abs(np.nanmin(ssh_init_s)), abs(np.nanmax(ssh_init_s)),
                   abs(np.nanmin(ssh_final_s)), abs(np.nanmax(ssh_final_s)))
    if has_hycom:
        vmax_ssh_h = max(abs(np.nanmin(hycom_ssh_s)), abs(np.nanmax(hycom_ssh_s)),
                         abs(np.nanmin(_smooth(hycom_init['ssh']))),
                         abs(np.nanmax(_smooth(hycom_init['ssh']))))
        vmax_ssh = (vmax_ssh + vmax_ssh_h) / 2.0

    vmax_u = max(abs(np.nanmin(u_init_s)), abs(np.nanmax(u_init_s)),
                 abs(np.nanmin(u_final_s)), abs(np.nanmax(u_final_s)))
    vmax_v = max(abs(np.nanmin(v_init_s)), abs(np.nanmax(v_init_s)),
                 abs(np.nanmin(v_final_s)), abs(np.nanmax(v_final_s)))
    if has_hycom:
        vmax_u = max(vmax_u, abs(np.nanmin(hycom_u_s)), abs(np.nanmax(hycom_u_s)))
        vmax_v = max(vmax_v, abs(np.nanmin(hycom_v_s)), abs(np.nanmax(hycom_v_s)))

    all_sst = np.concatenate([sst_init_s.compressed(), sst_final_s.compressed()])
    if has_hycom:
        all_sst = np.concatenate([all_sst, hycom_sst_s.compressed()])
    sst_vmin, sst_vmax = np.percentile(all_sst, 2), np.percentile(all_sst, 98)

    im_ssh0 = _plot(axes[0, 0], ssh_init_s,  f'{label} SSH (Initial)', 'RdBu_r', -vmax_ssh, vmax_ssh)
    im_ssh1 = _plot(axes[0, 1], ssh_final_s, f'{label} SSH (Final)',   'RdBu_r', -vmax_ssh, vmax_ssh)
    im_u0 = _plot(axes[1, 0], u_init_s,  f'{label} U (Initial)', 'RdBu_r', -vmax_u, vmax_u)
    im_u1 = _plot(axes[1, 1], u_final_s, f'{label} U (Final)',   'RdBu_r', -vmax_u, vmax_u)
    im_v0 = _plot(axes[2, 0], v_init_s,  f'{label} V (Initial)', 'RdBu_r', -vmax_v, vmax_v)
    im_v1 = _plot(axes[2, 1], v_final_s, f'{label} V (Final)',   'RdBu_r', -vmax_v, vmax_v)
    im_sst0 = _plot(axes[3, 0], sst_init_s,  f'{label} SST (Initial)', 'RdYlBu_r', sst_vmin, sst_vmax)
    im_sst1 = _plot(axes[3, 1], sst_final_s, f'{label} SST (Final)',   'RdYlBu_r', sst_vmin, sst_vmax)

    if has_hycom:
        _plot(axes[0, 2], hycom_ssh_s, 'HYCOM SSH (Final)', 'RdBu_r', -vmax_ssh, vmax_ssh)
        _plot(axes[1, 2], hycom_u_s,   'HYCOM U (Final)',   'RdBu_r', -vmax_u, vmax_u)
        _plot(axes[2, 2], hycom_v_s,   'HYCOM V (Final)',   'RdBu_r', -vmax_v, vmax_v)
        _plot(axes[3, 2], hycom_sst_s, 'HYCOM SST (Final)', 'RdYlBu_r', sst_vmin, sst_vmax)

    for r, (im, cbl) in enumerate(zip([im_ssh1, im_u1, im_v1, im_sst1],
                                       ['m', 'm/s', 'm/s', '$^\\circ$C'])):
        fig.colorbar(im, cax=cbar_axes[r], label=cbl)
    savefig(fig, fname)

print("plot_field_panels defined.")

## 4c. Plotting — Observation Scatter Map

Scatter plot of drifter and SWOT satellite observation positions.

In [ ]:
def plot_obs_scatter(obs_nc_path, fname, ny=70, nx=80):
    """Scatter plot of drifter + SWOT observation positions."""
    from matplotlib.lines import Line2D
    lon = np.linspace(LON_RANGE[0], LON_RANGE[1], nx)
    lat = np.linspace(LAT_RANGE[0], LAT_RANGE[1], ny)
    ncells = ny * nx

    v1_nc = os.path.join(BASEDIR, 'output_lsmcmc_ldata_V1', 'mlswe_lsmcmc_out.nc')
    H_b = None
    if os.path.exists(v1_nc):
        with Dataset(v1_nc) as nc:
            H_b = np.asarray(nc.variables['H_b'][:]).reshape(ny, nx)

    with Dataset(obs_nc_path) as nc:
        yind = np.asarray(nc.variables['yobs_ind_all'][:])
    nassim = yind.shape[0]

    def get_drifter_lonlat(cycle):
        ind1 = yind[cycle]; valid = ind1 >= 0
        u_inds = ind1[valid]
        u_mask = (u_inds >= ncells) & (u_inds < 2 * ncells)
        cell = u_inds[u_mask] - ncells
        return lon[cell % nx], lat[cell // nx]

    swot_dirs = [os.path.join(BASEDIR, d)
                 for d in ['data/swot_2024aug_new', 'data/swot_2024aug', 'data/swot']]
    swot_files = []
    for sd in swot_dirs:
        swot_files = sorted(glob.glob(os.path.join(sd, 'SWOT_*.nc')))
        if swot_files: break

    swot_lon1, swot_lat1, swot_lon2, swot_lat2 = [], [], [], []
    for fi, sf in enumerate(swot_files):
        try:
            with Dataset(sf) as nc:
                slat = np.array(nc.variables['latitude'][:])
                slon = np.array(nc.variables['longitude'][:])
            slon = np.where(slon > 180, slon - 360, slon)
            valid = (np.isfinite(slon.ravel()) & np.isfinite(slat.ravel()) &
                     (slon.ravel() >= -60) & (slon.ravel() <= -20) &
                     (slat.ravel() >= 10) & (slat.ravel() <= 45))
            lv, av = slon.ravel()[valid], slat.ravel()[valid]
            if fi < 3: swot_lon1.extend(lv); swot_lat1.extend(av)
            if fi >= len(swot_files) - 3: swot_lon2.extend(lv); swot_lat2.extend(av)
        except Exception:
            pass

    fig, ax = plt.subplots(figsize=(10, 7))
    if H_b is not None:
        lon2d, lat2d = np.meshgrid(lon, lat)
        ax.contourf(lon2d, lat2d, H_b, levels=20, cmap='Blues', alpha=0.4)
    if swot_lon1: ax.scatter(swot_lon1, swot_lat1, s=1, c='green', alpha=0.3, rasterized=True)
    if swot_lon2: ax.scatter(swot_lon2, swot_lat2, s=1, c='orange', alpha=0.3, rasterized=True)
    dl1, da1 = get_drifter_lonlat(0)
    ax.scatter(dl1, da1, s=18, c='red', edgecolors='white', linewidths=0.3, zorder=5)
    dl2, da2 = (get_drifter_lonlat(nassim - 1) if nassim > 1
                else (np.array([]), np.array([])))
    if len(dl2): ax.scatter(dl2, da2, s=18, c='blue', edgecolors='white',
                            linewidths=0.3, alpha=0.7, zorder=5)

    elems = [Line2D([0],[0], marker='o', color='w', markerfacecolor='red',
                    markeredgecolor='white', markersize=8,
                    label=f'Drifters Cycle 1 ({len(dl1)})')]
    if len(dl2): elems.append(
        Line2D([0],[0], marker='o', color='w', markerfacecolor='blue',
               markeredgecolor='white', markersize=8,
               label=f'Drifters Cycle {nassim} ({len(dl2)})'))
    if swot_lon1: elems.append(
        Line2D([0],[0], marker='s', color='w', markerfacecolor='green', markersize=10,
               label='SWOT first passes'))
    if swot_lon2: elems.append(
        Line2D([0],[0], marker='s', color='w', markerfacecolor='orange', markersize=10,
               label='SWOT last passes'))
    ax.legend(handles=elems, fontsize=12, loc='upper right', frameon=True, fancybox=True)
    ax.set_title('Observation Sources', fontsize=18)
    ax.set_xlabel('Longitude ($^\\circ$E)'); ax.set_ylabel('Latitude ($^\\circ$N)')
    fig.tight_layout()
    savefig(fig, fname)

print("plot_obs_scatter defined.")

---
## Figure 1 — Localization Illustration (V1 vs V2)

This figure is fully self-contained (no data files needed). It illustrates the two localization variants on a synthetic grid with random observations.

In [ ]:
# ── Figure 1: Localization V1 vs V2 illustration ──
# Run the self-contained script (uses np.random.seed(42), no external data)
%run generate_localization_illustration.py
# Display the generated figure
display(Image(filename=os.path.join(OUTDIR, 'localization_v1_v2_illustration.png'), width=900))

---
## Figures 2–6 — Linear Gaussian Experiment ($d = 14{,}400$)

**Data:** Self-contained. Run `linear_forward_generate_data.py` once to create `linear_gaussian_data.npz`, then run the KF/LSMCMC/LETKF scripts. All results are read from `linear_gaussian/output_*/`.

These figures use pre-computed results from the `linear_gaussian/` directory:
- **Fig 2** (`lg_obs_swaths.pdf`): SWOT-like swath observation pattern
- **Fig 3** (`lg_letkf_sensitivity.pdf`): LETKF sensitivity heatmap
- **Fig 4** (`lg_rmse_timeseries.pdf`): RMSE against exact KF over 100 cycles
- **Fig 5** (`lg_coord50.pdf`): Time series at the most-observed grid point
- **Fig 6** (`lg_snapshot.pdf`): Spatial analysis fields at cycles 50 and 100

In [ ]:
# ── Load Linear Gaussian data ──────────────────────────────────────
lgdir = os.path.join(BASEDIR, 'linear_gaussian')

# Check if data files exist (skip gracefully if not)
lg_data_path = os.path.join(lgdir, 'linear_gaussian_data.npz')
lg_available = os.path.exists(lg_data_path)

if lg_available:
    dat = np.load(lg_data_path)
    Z_truth  = dat['Z_truth']
    T_lg     = int(dat['T'])
    d_lg     = int(dat['d'])
    Ngx, Ngy = int(dat['Ngx']), int(dat['Ngy'])
    obs_inds = dat['obs_inds']       # (T, n_obs_per_cycle)
    
    kf_mean  = np.load(os.path.join(lgdir, 'linear_gaussian_kf.npz'))['kf_mean']
    
    # M=1 filters
    v1_m1 = np.load(os.path.join(lgdir, 'linear_gaussian_lsmcmc_v1.npz'))['lsmcmc_mean']
    v2_m1 = np.load(os.path.join(lgdir, 'linear_gaussian_lsmcmc_v2.npz'))['lsmcmc_mean']
    letkf_mean = np.load(os.path.join(lgdir, 'linear_gaussian_letkf_K50.npz'))['letkf_mean']
    
    # M=4 averaged filters
    v1_m4 = np.load(os.path.join(lgdir, 'linear_gaussian_lsmcmc_v1_avg.npz'))['lsmcmc_mean']
    v2_m4 = np.load(os.path.join(lgdir, 'linear_gaussian_lsmcmc_v2_avg.npz'))['lsmcmc_mean']
    
    # LETKF sensitivity results (optional — needed only for Figure 3)
    letkf_sens_path = os.path.join(lgdir, 'letkf_sensitivity_results.json')
    if os.path.exists(letkf_sens_path):
        with open(letkf_sens_path, 'r') as f:
            letkf_sens = json.load(f)
    else:
        letkf_sens = None
        print("  Note: letkf_sensitivity_results.json not found — Figure 3 will be skipped.")
    
    print(f"Linear Gaussian: d={d_lg}, grid={Ngx}x{Ngy}, T={T_lg}")
    print(f"  KF shape:     {kf_mean.shape}")
    print(f"  V1 M=1 shape: {v1_m1.shape}")
    print(f"  LETKF shape:  {letkf_mean.shape}")
else:
    print("Linear Gaussian data not found. Skipping Figures 2-6.")
    print("To generate this data, run: cd linear_gaussian && python3 run_full_comparison.py")
    T_lg, d_lg, Ngx, Ngy = None, None, None, None
    kf_mean, v1_m1, v2_m1, letkf_mean = None, None, None, None
    v1_m4, v2_m4, letkf_sens, obs_inds = None, None, None, None

In [ ]:
# ── Figure 2: SWOT-like observation swaths (2×2 panel) ─────────────
if lg_available and obs_inds is not None:
    fig, axes = plt.subplots(2, 2, figsize=(10, 10))
    for idx, cyc in enumerate([0, 3, 7, 15]):
        r, c = divmod(idx, 2)
        if cyc < len(obs_inds):
            obs_i = obs_inds[cyc]
            grid = np.zeros(d_lg)
            grid[obs_i] = 1.0
            axes[r, c].imshow(grid.reshape(Ngy, Ngx), origin='lower', cmap='Blues', vmin=0, vmax=1)
            axes[r, c].set_title(f'Cycle {cyc + 1} ({len(obs_i)} obs)', fontsize=18)
            axes[r, c].set_xlabel('$x$', fontsize=16)
            axes[r, c].set_ylabel('$y$', fontsize=16)
            axes[r, c].tick_params(labelsize=14)
    fig.suptitle('SWOT-like Swath Observation Pattern', fontsize=20, y=1.02)
    fig.tight_layout()
    savefig(fig, 'lg_obs_swaths.pdf')
    display(Image(filename=os.path.join(OUTDIR, 'lg_obs_swaths.png')))
else:
    print("Skipped Figure 2: Linear Gaussian data not available.")

In [ ]:
# ── Figure 3: LETKF sensitivity heatmap ────────────────────────────
if lg_available and letkf_sens is not None:
    hscales = sorted(set(d['hscale'] for d in letkf_sens['results']))
    alphas  = sorted(set(d['covinflate1'] for d in letkf_sens['results']))
    nh, na = len(hscales), len(alphas)
    
    rmse_grid = np.full((nh, na), np.nan)
    lookup = {(d['hscale'], d['covinflate1']): d['rmse'] for d in letkf_sens['results']}
    for i, h in enumerate(hscales):
        for j, a in enumerate(alphas):
            if (h, a) in lookup:
                rmse_grid[i, j] = lookup[(h, a)]
    
    fig, ax = plt.subplots(figsize=(8, 5.5))
    im = ax.imshow(rmse_grid, origin='lower', aspect='auto', cmap='viridis_r',
                   vmin=np.nanmin(rmse_grid), vmax=np.nanpercentile(rmse_grid, 95))
    ax.set_xticks(range(na))
    ax.set_xticklabels([f'{a:.2f}' for a in alphas], rotation=45, ha='right', fontsize=10)
    ax.set_yticks(range(nh))
    ax.set_yticklabels([f'{h:g}' for h in hscales], fontsize=11)
    ax.set_xlabel(r'Inflation factor $\alpha$')
    ax.set_ylabel(r'Localization radius $h_{\mathrm{loc}}$')
    ax.set_title(r'LETKF RMSE vs KF ($K{=}50$)')
    cb = plt.colorbar(im, ax=ax, shrink=0.85)
    cb.set_label('RMSE vs KF')
    
    best = min(letkf_sens['results'], key=lambda d: d['rmse'])
    bi = hscales.index(best['hscale'])
    bj = alphas.index(best['covinflate1'])
    ax.plot(bj, bi, marker='*', markersize=18, color='white',
            markeredgecolor='black', markeredgewidth=1.0)
    ax.annotate(f'{best["rmse"]:.4f}', xy=(bj, bi), xytext=(bj + 1.2, bi + 0.8),
                fontsize=10, color='white', fontweight='bold',
                arrowprops=dict(arrowstyle='->', color='white', lw=1.5))
    fig.tight_layout()
    savefig(fig, 'lg_letkf_sensitivity.pdf')
    display(Image(filename=os.path.join(OUTDIR, 'lg_letkf_sensitivity.png')))
else:
    print("Skipped Figure 3: LETKF sensitivity data not available.")
    print("To generate: cd linear_gaussian && python3 linear_forward_run_letkf_sensitivity.py")

In [ ]:
# ── Figure 4: RMSE vs KF time series ───────────────────────────────
colors_lg = {'V1': COLORS['V1'], 'V2': COLORS['V2'], 'LETKF': COLORS['LETKF']}

fig, ax = plt.subplots(figsize=(10, 5))
# M=1 filters
for key, fmean, label in [('V1', v1_m1, 'LSMCMC V1'), ('V2', v2_m1, 'LSMCMC V2')]:
    rmse = np.sqrt(np.mean((fmean - kf_mean)**2, axis=1))
    ax.plot(range(T_lg + 1), rmse, label=f'{label} ($M{{=}}1$)',
            color=colors_lg[key], linewidth=2, linestyle='-')
# M=4 averaged filters
for key, fmean, label in [('V1', v1_m4, 'LSMCMC V1'), ('V2', v2_m4, 'LSMCMC V2')]:
    rmse = np.sqrt(np.mean((fmean - kf_mean)**2, axis=1))
    ax.plot(range(T_lg + 1), rmse, label=f'{label} ($M{{=}}4$)',
            color=colors_lg[key], linewidth=2, linestyle='--')
# LETKF
rmse = np.sqrt(np.mean((letkf_mean - kf_mean)**2, axis=1))
ax.plot(range(T_lg + 1), rmse, label='LETKF ($K{=}50$)',
        color=colors_lg['LETKF'], linewidth=2)
ax.set_xlabel('Assimilation Cycle')
ax.set_ylabel('RMSE vs Kalman Filter')
ax.set_title('RMSE against Exact KF Solution')
ax.legend(frameon=True, fancybox=True, shadow=False)
ax.grid(True, alpha=0.3)
fig.tight_layout()
savefig(fig, 'lg_rmse_timeseries.pdf')
    plt.show()


In [ ]:
# ── Figure 5: Time series at most-observed coordinate ──────────────
obs_counts = np.zeros(d_lg, dtype=int)
for k in range(T_lg):
    oi = obs_inds[k]
    valid = oi[oi >= 0]
    obs_counts[valid] += 1
coord = int(np.argmax(obs_counts))
n_obs_coord = obs_counts[coord]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(range(T_lg + 1), kf_mean[:, coord], color='gray', label='KF', linewidth=2)
for key, fmean, label in [('V1', v1_m4, 'LSMCMC V1'), ('V2', v2_m4, 'LSMCMC V2')]:
    ax.plot(range(T_lg + 1), fmean[:, coord], label=f'{label} ($M{{=}}4$)',
            color=colors_lg[key], linewidth=1.5)
ax.plot(range(T_lg + 1), letkf_mean[:, coord], label='LETKF ($K{=}50$)',
        color=colors_lg['LETKF'], linewidth=1.5)
ax.set_xlabel('Assimilation Cycle')
ax.set_ylabel('State Value')
ax.set_title(f'Time Series at Most-Observed Grid Point ({coord}, observed {n_obs_coord}/{T_lg} cycles)')
ax.legend(frameon=True, fancybox=True, ncol=2)
ax.grid(True, alpha=0.3)
fig.tight_layout()
savefig(fig, 'lg_coord50.pdf')
    plt.show()


In [ ]:
# ── Figure 6: Snapshot at cycles 50 and T (4×3 grid) ──────────────
fig, axes = plt.subplots(4, 3, figsize=(15, 18))
cycle_mid, cycle_final = 50, T_lg

filters_m1 = {'LSMCMC V1': v1_m1, 'LSMCMC V2': v2_m1}
filters_m4 = {'LSMCMC V1': v1_m4, 'LSMCMC V2': v2_m4}

snapshot_grid = [
    (0, 0, kf_mean,     'KF'),
    (0, 1, v1_m1,       'V1 ($M{=}1$)'),
    (0, 2, v2_m1,       'V2 ($M{=}1$)'),
    (1, 0, letkf_mean,  'LETKF ($K{=}50$)'),
    (1, 1, v1_m4,       'V1 ($M{=}4$)'),
    (1, 2, v2_m4,       'V2 ($M{=}4$)'),
    (2, 0, kf_mean,     'KF'),
    (2, 1, v1_m1,       'V1 ($M{=}1$)'),
    (2, 2, v2_m1,       'V2 ($M{=}1$)'),
    (3, 0, letkf_mean,  'LETKF ($K{=}50$)'),
    (3, 1, v1_m4,       'V1 ($M{=}4$)'),
    (3, 2, v2_m4,       'V2 ($M{=}4$)'),
]

for block_idx, cyc in enumerate([cycle_mid, cycle_final]):
    row_offset = block_idx * 2
    kf_grid = kf_mean[cyc].reshape(Ngy, Ngx)
    vmin, vmax = np.nanmin(kf_grid), np.nanmax(kf_grid)
    for r, c, arr, lbl in snapshot_grid:
        if r < row_offset or r >= row_offset + 2:
            continue
        grid = arr[cyc].reshape(Ngy, Ngx) if arr is not None else np.zeros((Ngy, Ngx))
        im = axes[r, c].imshow(grid, origin='lower', vmin=vmin, vmax=vmax, cmap='RdBu_r')
        axes[r, c].set_title(f'{lbl} (Cycle {cyc})')
        plt.colorbar(im, ax=axes[r, c], shrink=0.8)

for ax in axes.ravel():
    ax.set_xlabel('$x$'); ax.set_ylabel('$y$')
fig.tight_layout()
savefig(fig, 'lg_snapshot.pdf')
    plt.show()


## Figures 7–9 — MLSWE Linear Observation Model

**Data:** Requires the `data/` folder (HYCOM BC, drifters, SWOT, bathymetry). If not already downloaded, run:
```bash
python3 download_all_data.py        # download raw data (~200–600 MB)
python3 generate_drifter_obs.py     # process drifter obs
python3 prebin_swot_ssh.py          # bin SWOT SSH
python3 generate_synthetic_swot.py  # fill empty SWOT cycles
```
Once `data/` is populated, the NL twin and Cauchy experiments below also have access to it — no additional downloads needed.

These figures use the multi-layer SWE model with **linear** observation operators (drifter velocity + SWOT SSH + SST).

- **Figure 7** (`mlswe_obs_pattern.pdf`): Drifter + SWOT observation scatter
- **Figure 8** (`ldata_compare_rmse.pdf`): 3-panel RMSE comparison (velocity / SST / SSH)
- **Figure 9** (`ldata_v1_fields.pdf`): SSH / U / V / SST field snapshots (initial + final + HYCOM)

In [ ]:
# ── Load MLSWE linear-obs analysis outputs ─────────────────────────
v1_nc  = os.path.join(BASEDIR, 'output_lsmcmc_ldata_V1', 'mlswe_lsmcmc_out.nc')
v2_nc  = os.path.join(BASEDIR, 'output_lsmcmc_ldata_V2', 'mlswe_lsmcmc_out.nc')
v1_obs = os.path.join(BASEDIR, 'output_lsmcmc_ldata_V1', 'mlswe_merged_obs.nc')

letkf_nc = None
for c in ['output_letkf_ldata_K25/mlswe_letkf_out.nc',
          'output_letkf_ldata/mlswe_letkf_out.nc',
          'output_letkf_best/mlswe_letkf_out.nc']:
    p = os.path.join(BASEDIR, c)
    if os.path.exists(p):
        letkf_nc = p; break

v1_ldata = load_mlswe_analysis(v1_nc) if os.path.exists(v1_nc) else None
v2_ldata = load_mlswe_analysis(v2_nc) if os.path.exists(v2_nc) else None
letkf_ldata = load_mlswe_analysis(letkf_nc) if letkf_nc else None

bc_path = _find_bc_path()
hycom_init_ld, hycom_final_ld = None, None
if bc_path and v1_ldata:
    hycom_init_ld, hycom_final_ld = _load_hycom_pair(bc_path, v1_ldata)

print("MLSWE linear-obs data loaded:")
for name, d in [('V1', v1_ldata), ('V2', v2_ldata), ('LETKF', letkf_ldata)]:
    if d:
        print(f"  {name}: {len(d['rmse_vel'])} cycles,  "
              f"vel={np.mean(d['rmse_vel']):.4f}  "
              f"sst={np.mean(d['rmse_sst']):.4f}")
    else:
        print(f"  {name}: not found")

In [ ]:
# ── Figure 7: Observation scatter (drifters + SWOT) ─────────────
if os.path.exists(v1_obs):
    plot_obs_scatter(v1_obs, 'mlswe_obs_pattern.pdf')
    plt.show()
else:
    print("Skipped: V1 obs file not found.")

# Note: SWOT swaths only appear if SWOT L2 files are present in data/swot_2024aug_new/
# The Zenodo archive contains pre-computed DA OUTPUT (analysis fields), not raw observation data.
# Raw SWOT L2 files require NASA Earthdata authentication and must be downloaded separately
# using download_all_data.py. On Binder, only drifter positions will be shown in this figure.


In [ ]:
# ── Figure 8: RMSE comparison (velocity / SST / SSH) ──────────────
datasets_ld, labels_ld, cols_ld = [], [], []
if v1_ldata: datasets_ld.append(v1_ldata); labels_ld.append('LSMCMC V1'); cols_ld.append(COLORS['V1'])
if v2_ldata: datasets_ld.append(v2_ldata); labels_ld.append('LSMCMC V2'); cols_ld.append(COLORS['V2'])
if letkf_ldata: datasets_ld.append(letkf_ldata); labels_ld.append('LETKF'); cols_ld.append(COLORS['LETKF'])

if datasets_ld:
    plot_compare_rmse_3panel(datasets_ld, labels_ld, cols_ld,
                              'ldata_compare_rmse.pdf',
                              title='Linear Data Model: RMSE Comparison')
    plt.show()
else:
    print("No MLSWE linear data available.")

In [ ]:
# ── Figure 9: V1 field panels (SSH / U / V / SST + HYCOM) ─────────
if v1_ldata:
    plot_field_panels(v1_ldata, 'LSMCMC V1', 'ldata_v1_fields.pdf',
                      hycom_init=hycom_init_ld, hycom_final=hycom_final_ld)
    plt.show()
else:
    print("Skipped: V1 linear data not found.")

## Figures 10–12 — Velocity / SST / SSH Timeseries (V2 Linear)

> **Data:** Same as Figures 7–9 (see data preparation above). No additional downloads needed.

These figures show per-grid-cell analysis vs. observed time series for the V2 linear-observation experiment.
They are generated by `regen_timeseries_1x2.py`, which reads from `output_lsmcmc_ldata_V2/`.

- **Figure 10** (`mlswe_results_vel_timeseries.png`): Velocity at most-observed cells
- **Figure 11** (`mlswe_results_sst_timeseries.png`): SST at most-observed cells
- **Figure 12** (`mlswe_results_ssh_timeseries.png`): SSH (SWOT) at most-observed cells

In [ ]:
# Run the timeseries script (it saves PNGs into output_lsmcmc_ldata_V2/plots/)
%run regen_timeseries_1x2.py

# Display the three timeseries figures
ts_dir = os.path.join(BASEDIR, 'output_lsmcmc_ldata_V2', 'plots')
for fname in ['mlswe_results_vel_timeseries.png',
              'mlswe_results_sst_timeseries.png',
              'mlswe_results_ssh_timeseries.png']:
    path = os.path.join(ts_dir, fname)
    if os.path.exists(path):
        print(f"\n{fname}")
        display(Image(filename=path))
    else:
        print(f"  Not found: {path}")

## Figures 13–14 — MLSWE Nonlinear Twin Experiment

> **Data:** Same downloaded data as Figures 7–9 (see data preparation above). The **twin** observations are generated internally by the runner scripts (`run_mlswe_lsmcmc_nldata_V1_twin.py`, etc.) — no extra data downloads are needed. You must run those scripts first to produce the output directories:
> ```bash
> python3 run_mlswe_lsmcmc_nldata_V1_twin.py      # → output_lsmcmc_nldata_twin_V1/
> python3 run_mlswe_lsmcmc_nldata_V1_twin.py --hmc # → output_lsmcmc_nldata_twin_V1_hmc/  (if applicable)
> python3 run_mlswe_lsmcmc_nldata_V2_twin.py       # → output_lsmcmc_nldata_twin_V2/
> python3 run_mlswe_letkf_nl_twin.py               # → output_letkf_nltwin/
> ```

These figures use the **nonlinear** observation model (arctan-transformed drifter velocities) with twin (synthetic) data.

- **Figure 13** (`nltwin_compare_rmse.pdf`): 3-panel RMSE comparison (V1 pCN / V1 HMC / V2 / LETKF)
- **Figure 14** (`nltwin_v1_fields.pdf`): SSH / U / V / SST field snapshots + HYCOM

In [ ]:
# ── Load NL twin analysis outputs ──────────────────────────────────
nltwin_v1_nc     = os.path.join(BASEDIR, 'output_lsmcmc_nldata_twin_V1',     'mlswe_lsmcmc_out.nc')
nltwin_v1_hmc_nc = os.path.join(BASEDIR, 'output_lsmcmc_nldata_twin_V1_hmc', 'mlswe_lsmcmc_out.nc')
nltwin_v2_nc     = os.path.join(BASEDIR, 'output_lsmcmc_nldata_twin_V2',     'mlswe_lsmcmc_out.nc')

nltwin_v1     = load_mlswe_analysis(nltwin_v1_nc)     if os.path.exists(nltwin_v1_nc)     else None
nltwin_v1_hmc = load_mlswe_analysis(nltwin_v1_hmc_nc) if os.path.exists(nltwin_v1_hmc_nc) else None
nltwin_v2     = load_mlswe_analysis(nltwin_v2_nc)     if os.path.exists(nltwin_v2_nc)     else None

# LETKF NL twin (from .nc or log)
letkf_twin = None
letkf_twin_nc  = os.path.join(BASEDIR, 'output_letkf_nl_twin', 'mlswe_letkf_out.nc')
letkf_twin_log = os.path.join(BASEDIR, 'letkf_nltwin_run.log')
if os.path.exists(letkf_twin_nc):
    letkf_twin = load_mlswe_analysis(letkf_twin_nc)
elif os.path.exists(letkf_twin_log):
    import re as _re
    _pat = _re.compile(r'\[(\d+)/\d+\].*vel=([\d.eE+-]+)\s+sst=([\d.]+)K\s+ssh=([\d.]+)m')
    _vel, _sst, _ssh = [], [], []
    with open(letkf_twin_log) as _f:
        for _line in _f:
            _m = _pat.search(_line)
            if _m: _vel.append(float(_m.group(2))); _sst.append(float(_m.group(3))); _ssh.append(float(_m.group(4)))
    if _vel:
        letkf_twin = {'rmse_vel': np.array(_vel), 'rmse_sst': np.array(_sst), 'rmse_ssh': np.array(_ssh)}

bc_path = _find_bc_path()
hycom_init_nt, hycom_final_nt = None, None
src = nltwin_v1_hmc if nltwin_v1_hmc else nltwin_v1
if bc_path and src:
    hycom_init_nt, hycom_final_nt = _load_hycom_pair(bc_path, src)

print("NL twin data loaded:")
for name, d in [('V1 pCN', nltwin_v1), ('V1 HMC', nltwin_v1_hmc),
                ('V2', nltwin_v2), ('LETKF', letkf_twin)]:
    if d:
        print(f"  {name}: vel={np.mean(d['rmse_vel']):.4f}  "
              f"sst={np.mean(d['rmse_sst']):.4f}")
    else:
        print(f"  {name}: not found")

In [ ]:
# ── Figure 13: NL twin RMSE comparison ─────────────────────────────
ds_nt, lb_nt, cl_nt = [], [], []
if nltwin_v1:     ds_nt.append(nltwin_v1);     lb_nt.append('LSMCMC V1 (pCN)'); cl_nt.append(COLORS['V1'])
if nltwin_v1_hmc: ds_nt.append(nltwin_v1_hmc); lb_nt.append('LSMCMC V1 (HMC)'); cl_nt.append('#9467bd')
if nltwin_v2:     ds_nt.append(nltwin_v2);     lb_nt.append('LSMCMC V2');        cl_nt.append(COLORS['V2'])
if letkf_twin:   ds_nt.append(letkf_twin);    lb_nt.append('LETKF');            cl_nt.append(COLORS['LETKF'])

if ds_nt:
    plot_compare_rmse_3panel(ds_nt, lb_nt, cl_nt, 'nltwin_compare_rmse.pdf',
                              title='NL Twin: RMSE Comparison',
                              sigma_ssh=0.50, ssh_clip=5.0)
    plt.show()
else:
    print("No NL twin data available.")

In [ ]:
# ── Figure 14: NL twin field panels (V1 HMC preferred) ────────────
field_data = nltwin_v1_hmc if nltwin_v1_hmc else nltwin_v1
field_label = 'LSMCMC V1 (HMC)' if nltwin_v1_hmc else 'LSMCMC V1'
if field_data:
    plot_field_panels(field_data, field_label, 'nltwin_v1_fields.pdf',
                      hycom_init=hycom_init_nt, hycom_final=hycom_final_nt)
    plt.show()
else:
    print("Skipped: NL twin V1 data not found.")

## Figures 15–19 — arctan + Cauchy Noise Twin Experiment

> **Data:** Same downloaded data as Figures 7–9 (see data preparation above). The Cauchy-noise twin observations are generated internally by the runner scripts in `nlgamma_ldata/`. Run them first:
> ```bash
> # Run all three variants sequentially (or use run_nlgamma_sequential.sh):
> cd nlgamma_ldata
> python3 run_nlgamma_ldata_V1.py       # → output_nlgamma_ldata_V1/
> python3 run_nlgamma_ldata_V1_hmc.py   # → output_nlgamma_ldata_V1_hmc/
> python3 run_nlgamma_ldata_V2.py       # → output_nlgamma_ldata_V2/
> cd ..
> ```

These figures use the **arctan + Cauchy** (non-Gaussian) observation model on synthetic twin data.
The standalone script `generate_nlgamma_figures.py` produces all five figures:

- **Figure 15** (`nlgamma_compare_rmse.pdf`): 3-panel RMSE comparison (V1 pCN / V1 HMC / V2)
- **Figure 16** (`nlgamma_v1_fields.pdf`): V1 field panels (SSH / U / V / SST + HYCOM)
- **Figure 17** (`nlgamma_v2_fields.pdf`): V2 M=1 field panels
- **Figure 18** (`nlgamma_mcmc_diag_v1_u.pdf`): V1 MCMC diagnostics (partition block)
- **Figure 19** (`nlgamma_mcmc_diag_v2.pdf`): V2 MCMC diagnostics (halo-localized block)

> **Note:** The MCMC diagnostic figures (18–19) re-run short MCMC chains (~2500 samples) and take an extra 1–2 minutes.

In [ ]:
# Run generate_nlgamma_figures.py as a subprocess to avoid
# matplotlib backend conflicts with %matplotlib inline.
import subprocess, sys

result = subprocess.run(
    [sys.executable, os.path.join(BASEDIR, 'generate_nlgamma_figures.py')],
    cwd=BASEDIR, capture_output=True, text=True, timeout=600)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)

In [ ]:
# Display Cauchy-noise figures
nlg_figs = [
    ('nlgamma_compare_rmse.png',    'Figure 15: RMSE Comparison'),
    ('nlgamma_v1_fields.png',       'Figure 16: V1 Field Panels'),
    ('nlgamma_v2_fields.png',       'Figure 17: V2 Field Panels'),
    ('nlgamma_mcmc_diag_v1_u.png',  'Figure 18: V1 MCMC Diagnostics'),
    ('nlgamma_mcmc_diag_v2.png',    'Figure 19: V2 MCMC Diagnostics'),
]

for fname, desc in nlg_figs:
    path = os.path.join(OUTDIR, fname)
    if os.path.exists(path):
        print(f"\n{desc}: {fname}")
        display(Image(filename=path))
    else:
        # Try PDF fallback (some figures may only exist as PDF)
        pdf_path = path.replace('.png', '.pdf')
        if os.path.exists(pdf_path):
            print(f"\n{desc}: (PNG not found, PDF exists at {pdf_path})")
        else:
            print(f"  Not found: {fname}")


## Summary

All paper figures have been regenerated. The table below lists every figure and its output file.

In [ ]:
# ── Verify all expected figures exist ──────────────────────────────
expected_figures = {
    'Fig 1':  'localization_v1_v2_illustration.pdf',
    'Fig 2':  'lg_obs_swaths.pdf',
    'Fig 3':  'lg_letkf_sensitivity.pdf',
    'Fig 4':  'lg_rmse_timeseries.pdf',
    'Fig 5':  'lg_coord50.pdf',
    'Fig 6':  'lg_snapshot.pdf',
    'Fig 7':  'mlswe_obs_pattern.pdf',
    'Fig 8':  'ldata_compare_rmse.pdf',
    'Fig 9':  'ldata_v1_fields.pdf',
    'Fig 10': 'mlswe_results_vel_timeseries.png',
    'Fig 11': 'mlswe_results_sst_timeseries.png',
    'Fig 12': 'mlswe_results_ssh_timeseries.png',
    'Fig 13': 'nltwin_compare_rmse.pdf',
    'Fig 14': 'nltwin_v1_fields.pdf',
    'Fig 15': 'nlgamma_compare_rmse.pdf',
    'Fig 16': 'nlgamma_v1_fields.pdf',
    'Fig 17': 'nlgamma_v2_fields.pdf',
    'Fig 18': 'nlgamma_mcmc_diag_v1_u.pdf',
    'Fig 19': 'nlgamma_mcmc_diag_v2.pdf',
}

ts_dir = os.path.join(BASEDIR, 'output_lsmcmc_ldata_V2', 'plots')
found, missing = 0, 0
print(f"{'Figure':<8} {'Status':<8} {'File'}")
print('-' * 60)
for label, fname in expected_figures.items():
    # Check paper_figures/ first, then the timeseries output dir
    if os.path.exists(os.path.join(OUTDIR, fname)):
        print(f'{label:<8} {"OK":<8} paper_figures/{fname}')
        found += 1
    elif os.path.exists(os.path.join(ts_dir, fname)):
        print(f'{label:<8} {"OK":<8} output_lsmcmc_ldata_V2/plots/{fname}')
        found += 1
    else:
        print(f'{label:<8} {"MISS":<8} {fname}')
        missing += 1

print(f'\n{found}/{len(expected_figures)} figures found, {missing} missing.')
if missing == 0:
    print('\nAll figures generated successfully!')

## Environment Information

The cell below records the Python version and key dependency versions used to produce these results. This information is critical for reproducibility (see [Rule 5](https://doi.org/10.1371/journal.pcbi.1007007)).

In [ ]:
import platform

print("=" * 60)
print("Environment Information")
print("=" * 60)
print(f"  Python:     {sys.version}")
print(f"  Platform:   {platform.platform()}")
print(f"  NumPy:      {np.__version__}")
print(f"  SciPy:      {__import__('scipy').__version__}")
print(f"  Matplotlib: {matplotlib.__version__}")
print(f"  netCDF4:    {__import__('netCDF4').__version__}")
print(f"  PyYAML:     {__import__('yaml').__version__}")
print("=" * 60)

## How to Cite

If you use this code, please cite the paper and the software repository:

**Paper:**
> Ruzayqat, H., Chipilski, H. G., & Knio, O. (2026). Two Localization Strategies for Sequential MCMC Data Assimilation with Applications to Nonlinear Non-Gaussian Geophysical Models.

**Software:**
> Ruzayqat, H. (2026). LSMCMC [Software]. GitHub. https://github.com/ruzayqat/LSMCMC

See `CITATION.cff` in the repository root for a machine-readable citation file.

## License

This notebook and the accompanying code are released under the **MIT License**. See `LICENSE` for details.
